In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from sklearn.preprocessing import RobustScaler
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D1 
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [7]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [8]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [9]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [10]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'OS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,8.83,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,19.73,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,13.27,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [11]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [16]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,DFS,DFS_event


In [17]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS'])]

# y 
y = clinical_train.loc[:, ['DFS', 'event_DFS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['DFS'], [10, 90])
times = np.arange(lower, upper)

In [18]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [19]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [20]:
clinical_test.rename(columns = {'DFS_event' : 'event_DFS'}, inplace = True)

In [21]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'event_DFS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['DFS', 'event_DFS']]
lower, upper = np.percentile(y_MAASTRO['DFS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_DFS'], y_MAASTRO['DFS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

# Feature Selection: RENT

In [22]:
# Choose features from the result of RENT 
selected_features = ["hpv_related",
"pack_years",
"uicc8_III-IV",
"oropharynx",
"cavum_oris",
"TLG"]

In [23]:
X_rent = X.loc[:, selected_features]
X_new = X_rent.copy()

In [24]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Standardization

In [25]:
# Copy the original X for later 
original_X = X.copy()

In [26]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [32]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [33]:
X_new

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,0.0,0.000000,0.0,1,0,86.228420
1,0.0,27.404795,0.0,0,0,7.040100
2,0.0,41.019178,1.0,0,1,83.569669
3,0.0,37.500000,0.0,0,0,5.567091
4,0.0,53.000000,0.0,0,0,16.150550
...,...,...,...,...,...,...
134,1.0,0.000000,0.0,1,0,26.280140
135,1.0,0.000000,1.0,1,0,101.754834
136,1.0,39.498630,0.0,1,0,66.273201
137,1.0,71.527397,1.0,1,0,71.832443


In [34]:
X_new_std

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,0.0,-0.747766,0.0,1,0,0.231096
1,0.0,0.159774,0.0,0,0,-0.387041
2,0.0,0.610629,1.0,0,1,0.210342
3,0.0,0.494088,0.0,0,0,-0.398539
4,0.0,1.007387,0.0,0,0,-0.315926
...,...,...,...,...,...,...
134,1.0,-0.747766,0.0,1,0,-0.236855
135,1.0,-0.747766,1.0,1,0,0.352293
136,1.0,0.560275,0.0,1,0,0.075327
137,1.0,1.620942,1.0,1,0,0.118722


In [35]:
MAASTRO_new 

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,1,0,0,1,0,263.611623
1,0,20,1,1,0,36.980700
2,0,6,1,1,0,74.636342
3,0,45,1,0,0,46.791979
4,1,59,0,1,0,107.637514
...,...,...,...,...,...,...
94,0,55,1,0,0,144.490782
95,0,174,1,0,0,69.214868
96,1,0,1,1,0,102.594274
97,1,0,0,1,0,103.229492


In [36]:
MAASTRO_new_std

,hpv_related,pack_years,uicc8_III-IV,oropharynx,cavum_oris,TLG
0,1,-0.747766,0,1,0,1.615732
1,0,-0.085444,1,1,0,-0.153327
2,0,-0.549070,1,1,0,0.140609
3,0,0.742458,1,0,0,-0.076742
4,1,1.206084,0,1,0,0.398213
...,...,...,...,...,...,...
94,0,1.073619,1,0,0,0.685886
95,0,5.014436,1,0,0,0.098289
96,1,-0.747766,1,1,0,0.358846
97,1,-0.747766,0,1,0,0.363804


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [37]:
# Setting the y format for skf below  
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:05:54,514] A new study created in memory with name: no-name-9d5c166a-8f66-4468-a513-b06ce9ad07dc


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-13 17:05:57,748] A new study created in memory with name: no-name-d35aed9f-d05e-40ae-ba12-700bc13b30b4


Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7446808510638298
Fold 4 C-index: 0.6768060836501901
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:05:57,741] Trial 0 finished with value: 0.6380043211913705 and parameters: {}. Best is trial 0 with value: 0.6380043211913705.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6380043211913705], datetime_start=datetime.datetime(2024, 4, 13, 17, 5, 54, 597507), datetime_complete=datetime.datetime(2024, 4, 13, 17, 5, 57, 741269), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6380043211913705


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23694732382610154
Fold 2 IBS: 0.2615158118717559
Fold 3 IBS: 0.17590965994359087
Fold 4 IBS: 0.28365588891632393
Fold 5 IBS: 0.19620686095559542
[I 2024-04-13 17:05:57,986] Trial 0 finished with value: 0.2308471091026735 and parameters: {}. Best is trial 0 with value: 0.2308471091026735.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2308471091026735], datetime_start=datetime.datetime(2024, 4, 13, 17, 5, 57, 785018), datetime_complete=datetime.datetime(2024, 4, 13, 17, 5, 57, 986232), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2308471091026735


In [38]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [39]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.638
train_ibs:  0.231


#### Test

In [40]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [41]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.532
IBS score: 0.276


In [42]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [43]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:05:58,153] A new study created in memory with name: no-name-d566a35a-304f-467c-a807-4242ecb30a95


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6374501992031872
Fold 2 C-index: 0.5348837209302325
Fold 3 C-index: 0.6085106382978723
Fold 4 C-index: 0.4543726235741445
Fold 5 C-index: 0.6781115879828327
[I 2024-04-13 17:05:58,299] Trial 0 finished with value: 0.5826657539976539 and parameters: {}. Best is trial 0 with value: 0.5826657539976539.


[I 2024-04-13 17:05:58,310] A new study created in memory with name: no-name-5641e89e-761a-4b03-9f06-07d17940d160




* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5826657539976539], datetime_start=datetime.datetime(2024, 4, 13, 17, 5, 58, 186454), datetime_complete=datetime.datetime(2024, 4, 13, 17, 5, 58, 297694), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.5826657539976539


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2472470947584011
Fold 2 IBS: 0.23203988427112568
Fold 3 IBS: 0.22898186356844594
Fold 4 IBS: 0.24197478455226282
Fold 5 IBS: 0.2293955853840956
[I 2024-04-13 17:05:58,494] Trial 0 finished with value: 0.23592784250686621 and parameters: {}. Best is trial 0 with value: 0.23592784250686621.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23592784250686621], datetime_start=datetime.datetime(2024, 4, 13, 17, 5, 58, 360178), datetime_complete=datetime.datetime(2024, 4, 13, 17, 5, 58, 494313), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23592784250686621


In [45]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.583
train_ibs:  0.236


#### Test

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.571


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.229


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [50]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:05:58,646] A new study created in memory with name: no-name-e8c0c485-a6c1-42ba-8e5d-159206d2be69


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923


[I 2024-04-13 17:05:58,929] A new study created in memory with name: no-name-b92377b2-fba8-432e-beea-a7f31a746f84


Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:05:58,923] Trial 0 finished with value: 0.6369089405587356 and parameters: {}. Best is trial 0 with value: 0.6369089405587356.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6369089405587356], datetime_start=datetime.datetime(2024, 4, 13, 17, 5, 58, 679566), datetime_complete=datetime.datetime(2024, 4, 13, 17, 5, 58, 923501), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6369089405587356


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.23861993031733617
Fold 2 IBS: 0.2596147966570129
Fold 3 IBS: 0.1774757774558778
Fold 4 IBS: 0.28226222126776673
Fold 5 IBS: 0.1964081480880089
[I 2024-04-13 17:05:59,199] Trial 0 finished with value: 0.23087617475720051 and parameters: {}. Best is trial 0 with value: 0.23087617475720051.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.23087617475720051], datetime_start=datetime.datetime(2024, 4, 13, 17, 5, 58, 952466), datetime_complete=datetime.datetime(2024, 4, 13, 17, 5, 59, 199691), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.23087617475720051


In [51]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [52]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.637
train_ibs:  0.231


#### Test

In [53]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [54]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.273


In [55]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [56]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:05:59,451] A new study created in memory with name: no-name-bc632e41-7f1c-440a-924a-6acb1e5c9a0f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:05:59,732] Trial 0 finished with value: 0.6369089405587356 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.6369089405587356.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.44866920152091255
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:05:59,930] Trial 1 finished with value: 0.593660845779121 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.6369089405587356.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.44866920152091255
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:06:00,161] Trial 2 finished with value: 0.593660845779121 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:06:04,731] Trial 24 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.39872004425394175}. Best is trial 5 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.44866920152091255
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:06:04,911] Trial 25 finished with value: 0.593660845779121 and parameters: {'l1_ratio': 0.2047855727725989}. Best is trial 5 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:06:05,170] Trial 26 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.3312072877416866}. Best is trial 5 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5

Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:06:10,114] Trial 48 finished with value: 0.6369089405587356 and parameters: {'l1_ratio': 0.5683872043393509}. Best is trial 5 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 17:06:10,326] Trial 49 finished with value: 0.6369089405587356 and parameters: {'l1_ratio': 0.6743720426532283}. Best is trial 5 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.6692015209125475
Fold 5 C-index: 0.6437768240343348
[I 2024-04-13 17:06:10,551] Trial 50 finished with value: 0.6377673096574481 and parameters: {'l1_ratio': 0.49233256947923487}. Best is trial 5 with value: 0.6377673096574481.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5155038759689923
Fold 3 C-index: 0.7

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.23844252456919102
Fold 2 IBS: 0.25962309452516086
Fold 3 IBS: 0.17756122520087697
Fold 4 IBS: 0.28222100167646247
Fold 5 IBS: 0.1963024150227557
[I 2024-04-13 17:06:21,206] Trial 0 finished with value: 0.23083005219888938 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.23083005219888938.
Fold 1 IBS: 0.23821227441764645
Fold 2 IBS: 0.259507281104852
Fold 3 IBS: 0.17748611076455306
Fold 4 IBS: 0.2711903560807151
Fold 5 IBS: 0.1961309053731986
[I 2024-04-13 17:06:21,423] Trial 1 finished with value: 0.22850538554819305 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.22850538554819305.
Fold 1 IBS: 0.2382318642303404
Fold 2 IBS: 0.2595942968590173
Fold 3 IBS: 0.1774558487970546
Fold 4 IBS: 0.2676448102491281
Fold 5 IBS: 0.19609818316114597
[I 2024-04-13 17:06:21,720] Trial 2 finished with value: 0.2278050006593373 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.2278050006593373.
F

Fold 1 IBS: 0.23823371554597988
Fold 2 IBS: 0.25949207642311956
Fold 3 IBS: 0.17750328264248502
Fold 4 IBS: 0.2822220524726955
Fold 5 IBS: 0.19617109028285518
[I 2024-04-13 17:06:27,420] Trial 25 finished with value: 0.23072444347342702 and parameters: {'l1_ratio': 0.3747482470228898}. Best is trial 22 with value: 0.22697662359136075.
Fold 1 IBS: 0.246428105921172
Fold 2 IBS: 0.23200855633752734
Fold 3 IBS: 0.22831992202640738
Fold 4 IBS: 0.2440469761668376
Fold 5 IBS: 0.22829284237284164
[I 2024-04-13 17:06:27,535] Trial 26 finished with value: 0.23581928056495718 and parameters: {'l1_ratio': 0.015423757551295547}. Best is trial 22 with value: 0.22697662359136075.
Fold 1 IBS: 0.23825230025286467
Fold 2 IBS: 0.2594664300976646
Fold 3 IBS: 0.17742775329061594
Fold 4 IBS: 0.26805651991682033
Fold 5 IBS: 0.1961032427289488
[I 2024-04-13 17:06:27,755] Trial 27 finished with value: 0.22786124925738288 and parameters: {'l1_ratio': 0.2329675843966648}. Best is trial 22 with value: 0.226976623

Fold 2 IBS: 0.25951865117334183
Fold 3 IBS: 0.17754931699399207
Fold 4 IBS: 0.28222615388072314
Fold 5 IBS: 0.19619038048128568
[I 2024-04-13 17:06:32,687] Trial 50 finished with value: 0.23074835144687372 and parameters: {'l1_ratio': 0.41653985858071785}. Best is trial 45 with value: 0.22681839843568477.
Fold 1 IBS: 0.2382357139485572
Fold 2 IBS: 0.2596101111858774
Fold 3 IBS: 0.17750899854071667
Fold 4 IBS: 0.26486067928576307
Fold 5 IBS: 0.1960820029155451
[I 2024-04-13 17:06:32,946] Trial 51 finished with value: 0.22725950117529187 and parameters: {'l1_ratio': 0.19031596268046638}. Best is trial 45 with value: 0.22681839843568477.
Fold 1 IBS: 0.23829655493110996
Fold 2 IBS: 0.25956288920813936
Fold 3 IBS: 0.17750653577127826
Fold 4 IBS: 0.26299168854643207
Fold 5 IBS: 0.19606915721735552
[I 2024-04-13 17:06:33,156] Trial 52 finished with value: 0.22688536513486302 and parameters: {'l1_ratio': 0.16901987791340187}. Best is trial 45 with value: 0.22681839843568477.
Fold 1 IBS: 0.2384

Fold 2 IBS: 0.25952472533669496
Fold 3 IBS: 0.1774960392545615
Fold 4 IBS: 0.2670682153268056
Fold 5 IBS: 0.19609819335409484
[I 2024-04-13 17:06:37,816] Trial 75 finished with value: 0.22768238769203286 and parameters: {'l1_ratio': 0.21877076880955348}. Best is trial 45 with value: 0.22681839843568477.
Fold 1 IBS: 0.23829998453371104
Fold 2 IBS: 0.23230675325916297
Fold 3 IBS: 0.22425800264666115
Fold 4 IBS: 0.2618485639188856
Fold 5 IBS: 0.1960657581178909
[I 2024-04-13 17:06:37,971] Trial 76 finished with value: 0.23055581249526233 and parameters: {'l1_ratio': 0.15699904383686278}. Best is trial 45 with value: 0.22681839843568477.
Fold 1 IBS: 0.23833259207918248
Fold 2 IBS: 0.23220358225358703
Fold 3 IBS: 0.22479972642178173
Fold 4 IBS: 0.2592480184868065
Fold 5 IBS: 0.1960543884558004
[I 2024-04-13 17:06:38,122] Trial 77 finished with value: 0.2301276615394316 and parameters: {'l1_ratio': 0.13190820156137945}. Best is trial 45 with value: 0.22681839843568477.
Fold 1 IBS: 0.23824614

In [57]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [58]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.638
train_ibs:  0.227


#### Test

In [59]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [60]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.4231641494784485)

test_cindex : 0.532


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.16531374004314667)

test_ibs:  0.273


In [61]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [62]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:06:42,664] A new study created in memory with name: no-name-25a4f96d-46d5-48a2-81cb-510ba651dfce


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6143410852713178
Fold 3 C-index: 0.6212765957446809
Fold 4 C-index: 0.6806083650190115
Fold 5 C-index: 0.575107296137339
[I 2024-04-13 17:06:44,514] Trial 0 finished with value: 0.6177885807850675 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.6177885807850675.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.624031007751938
Fold 3 C-index: 0.6595744680851063
Fold 4 C-index: 0.7053231939163498
Fold 5 C-index: 0.6030042918454935
[I 2024-04-13 17:06:45,923] Trial 1 finished with value: 0.6402989429173871 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'm

Fold 1 C-index: 0.6235059760956175
Fold 2 C-index: 0.6686046511627907
Fold 3 C-index: 0.7212765957446808
Fold 4 C-index: 0.7243346007604563
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 17:07:00,776] Trial 16 finished with value: 0.6771580986582885 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 17, 'max_depth': 7, 'n_estimators': 137, 'oob_score': True, 'max_samples': 0.962365055520953, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19037781478052387, 'warm_start': True}. Best is trial 14 with value: 0.7002369632470599.
Fold 1 C-index: 0.6294820717131474
Fold 2 C-index: 0.6686046511627907
Fold 3 C-index: 0.7085106382978723
Fold 4 C-index: 0.7091254752851711
Fold 5 C-index: 0.6416309012875536
[I 2024-04-13 17:07:01,333] Trial 17 finished with value: 0.6714707475493069 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 19, 'max_depth': 7, 'n_estimators': 180, 'oob_score': True, 'max_samples': 0.8699768148803853,

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.6666666666666666
Fold 3 C-index: 0.7787234042553192
Fold 4 C-index: 0.7357414448669202
Fold 5 C-index: 0.7017167381974249
[I 2024-04-13 17:07:06,136] Trial 31 finished with value: 0.7000756268928836 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 17, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 171, 'oob_score': True, 'max_samples': 0.5019192620095752, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.050771418210883164, 'warm_start': True}. Best is trial 29 with value: 0.7116053815973882.
Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.686046511627907
Fold 3 C-index: 0.7829787234042553
Fold 4 C-index: 0.7718631178707225
Fold 5 C-index: 0.7381974248927039
[I 2024-04-13 17:07:06,814] Trial 32 finished with value: 0.7177295061567273 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 223, 'oob_score': True, 'max_samples': 0.563642711374241

...

Fold 5 IBS: 0.2108108318582224
[I 2024-04-13 17:09:08,670] Trial 31 finished with value: 0.2206877222460263 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 9, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 396, 'oob_score': True, 'max_samples': 0.2868712535426494, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.021231453689495153}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.24659836091991297
Fold 2 IBS: 0.23212875696426277
Fold 3 IBS: 0.23013831975076382
Fold 4 IBS: 0.24109773536375373
Fold 5 IBS: 0.22997643124737513
[I 2024-04-13 17:09:11,321] Trial 32 finished with value: 0.23598792084921366 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 3, 'n_estimators': 323, 'oob_score': True, 'max_samples': 0.15457524910822473, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.004932622259272009}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.23197141083929648
Fold 2 IBS: 0.22

Fold 1 IBS: 0.23103878287252402
Fold 2 IBS: 0.22628918231285983
Fold 3 IBS: 0.2133979486579942
Fold 4 IBS: 0.22430579562693312
Fold 5 IBS: 0.21153361401748674
[I 2024-04-13 17:10:04,667] Trial 47 finished with value: 0.22131306469755957 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 8, 'min_samples_leaf': 11, 'max_depth': 10, 'n_estimators': 475, 'oob_score': False, 'max_samples': 0.43491671167283275, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.07826676341691717}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.2461442212529148
Fold 2 IBS: 0.23226781881803804
Fold 3 IBS: 0.22981312479103597
Fold 4 IBS: 0.24139414747597257
Fold 5 IBS: 0.2304782442201671
[I 2024-04-13 17:10:08,433] Trial 48 finished with value: 0.23601951131162568 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 3, 'min_samples_leaf': 19, 'max_depth': 6, 'n_estimators': 447, 'oob_score': False, 'max_samples': 0.2751094610735481, 'max_features': 'auto', 'min_weight_fraction_

Fold 5 IBS: 0.23018854152461504
[I 2024-04-13 17:10:53,195] Trial 62 finished with value: 0.23591814060376043 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 5, 'min_samples_leaf': 5, 'max_depth': 10, 'n_estimators': 455, 'oob_score': True, 'max_samples': 0.4677262455345284, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.2827349986373208}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.24652405780792855
Fold 2 IBS: 0.2322152403880562
Fold 3 IBS: 0.22946718005561817
Fold 4 IBS: 0.2414651127800749
Fold 5 IBS: 0.23031058287471998
[I 2024-04-13 17:10:57,861] Trial 63 finished with value: 0.23599643478127957 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 8, 'min_samples_leaf': 14, 'max_depth': 9, 'n_estimators': 499, 'oob_score': True, 'max_samples': 0.36524236153301975, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.19147391488941964}. Best is trial 26 with value: 0.22015707552217512.
Fold 1 IBS: 0.23373423023692633
Fold 2 IBS: 0.2252

Fold 1 IBS: 0.24217247193439537
Fold 2 IBS: 0.2278793635566358
Fold 3 IBS: 0.20080078124083145
Fold 4 IBS: 0.22310960096298033
Fold 5 IBS: 0.2079672399605948
[I 2024-04-13 17:11:56,152] Trial 78 finished with value: 0.22038589153108754 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 13, 'n_estimators': 418, 'oob_score': False, 'max_samples': 0.7742508336660989, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.033857990321958055}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.2355555064593619
Fold 2 IBS: 0.22224942419207222
Fold 3 IBS: 0.20877935998830383
Fold 4 IBS: 0.22444536984656233
Fold 5 IBS: 0.2081228365598182
[I 2024-04-13 17:11:59,557] Trial 79 finished with value: 0.21983049940922367 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 14, 'n_estimators': 439, 'oob_score': False, 'max_samples': 0.8561349620215079, 'max_features': 'log2', 'min_weight_fraction_le

Fold 5 IBS: 0.2069805372099246
[I 2024-04-13 17:13:02,085] Trial 93 finished with value: 0.21836991119421248 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 15, 'n_estimators': 436, 'oob_score': False, 'max_samples': 0.7876680525303928, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.02818336790921719}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.2481959844323671
Fold 2 IBS: 0.23173657169109757
Fold 3 IBS: 0.20201510994966232
Fold 4 IBS: 0.22255626189257152
Fold 5 IBS: 0.21061654425734486
[I 2024-04-13 17:13:06,288] Trial 94 finished with value: 0.22302409444460863 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 5, 'min_samples_leaf': 3, 'max_depth': 16, 'n_estimators': 436, 'oob_score': False, 'max_samples': 0.8362507108284499, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.027504161893380293}. Best is trial 71 with value: 0.21814006458981802.
Fold 1 IBS: 0.23915650996574608
Fold 2 IBS: 0.223

In [63]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [64]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.79
train_ibs:  0.218


#### Test

In [65]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [66]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=20, max_features='auto', max_leaf_nodes=12,
                     max_samples=0.9937352645069003, min_samples_leaf=1,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.014733302673387358,
                     n_estimators=435, random_state=123, warm_start=True)

test_cindex:  0.603


RandomSurvivalForest(max_depth=13, max_features='log2', max_leaf_nodes=3,
                     max_samples=0.785271119394671, min_samples_leaf=4,
                     min_samples_split=2,
                     min_weight_fraction_leaf=8.074256882937306e-05,
                     n_estimators=444, random_state=123)

test_ibs:  0.215


In [67]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [68]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [69]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:13:33,356] A new study created in memory with name: no-name-aac3f577-d5fc-403c-ba2e-77712697c30e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5976095617529881
Fold 2 C-index: 0.6666666666666666
Fold 3 C-index: 0.7382978723404255
Fold 4 C-index: 0.7319391634980988
Fold 5 C-index: 0.6909871244635193
[I 2024-04-13 17:13:34,352] Trial 0 finished with value: 0.6851000777443397 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.6851000777443397.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:13:36,927] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.6589147286821705
Fold 3 C-index: 0.7085106382978723
Fold 4 C-index: 0.6977186311787072
Fold 5 C-index: 0.6094420600858369
[I 2024-04-13 17:14:03,342] Trial 16 finished with value: 0.656032749497523 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.6881534529891422.
Fold 1 C-index: 0.5717131474103586
Fold 2 C-index: 0.6705426356589147
Fold 3 C-index: 0.7510638297872341
Fold 4 C-index: 0.7167300380228137
Fold 5 C-index: 0.6223175965665236
[I 2024-04-13 17:14:04,173] Trial 17 finished with value: 0.6664734494891689 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_featu

Fold 1 C-index: 0.6095617529880478
Fold 2 C-index: 0.6550387596899225
Fold 3 C-index: 0.7404255319148936
Fold 4 C-index: 0.7186311787072244
Fold 5 C-index: 0.7017167381974249
[I 2024-04-13 17:14:22,182] Trial 31 finished with value: 0.6850747922995026 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 5, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 400, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.6357896182918609, 'min_weight_fraction_leaf': 0.043457520259011145}. Best is trial 23 with value: 0.6906324467750945.
Fold 1 C-index: 0.6215139442231076
Fold 2 C-index: 0.6589147286821705
Fold 3 C-index: 0.7468085106382979
Fold 4 C-index: 0.7224334600760456
Fold 5 C-index: 0.6244635193133047
[I 2024-04-13 17:14:23,491] Trial 32 finished with value: 0.6748268325865853 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 412, 'oob_score': False, 'warm_start': True, 'max_features': 1

...

Fold 1 IBS: 0.23738806550248728
Fold 2 IBS: 0.21993578889192347
Fold 3 IBS: 0.21089498091706652
Fold 4 IBS: 0.2272048730326933
Fold 5 IBS: 0.2132773803587104
[I 2024-04-13 17:17:46,524] Trial 30 finished with value: 0.2217402177405762 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 13, 'max_depth': 9, 'n_estimators': 206, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.7311100810851326, 'min_weight_fraction_leaf': 0.16969817682654836}. Best is trial 20 with value: 0.21858490307862474.
Fold 1 IBS: 0.2371472983744441
Fold 2 IBS: 0.2192317816045922
Fold 3 IBS: 0.2066273024000569
Fold 4 IBS: 0.2223268600858535
Fold 5 IBS: 0.20805506169911736
[I 2024-04-13 17:17:50,411] Trial 31 finished with value: 0.21867766083281284 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 17, 'max_depth': 3, 'n_estimators': 259, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.84794250

Fold 1 IBS: 0.24647150063770518
Fold 2 IBS: 0.2324005238674857
Fold 3 IBS: 0.2295165680261027
Fold 4 IBS: 0.2413350176086155
Fold 5 IBS: 0.23048981014618952
[I 2024-04-13 17:18:50,498] Trial 45 finished with value: 0.2360426840572197 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 18, 'max_depth': 8, 'n_estimators': 184, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.42889080150309455, 'min_weight_fraction_leaf': 0.30395190085793466}. Best is trial 33 with value: 0.21855367964170153.
Fold 1 IBS: 0.23684554782082465
Fold 2 IBS: 0.21919333404974234
Fold 3 IBS: 0.2053013505980981
Fold 4 IBS: 0.22292477589597334
Fold 5 IBS: 0.20836678612341789
[I 2024-04-13 17:18:54,390] Trial 46 finished with value: 0.21852635889761127 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 6, 'n_estimators': 273, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0

Fold 1 IBS: 0.2427874690307322
Fold 2 IBS: 0.22051307489638503
Fold 3 IBS: 0.21103171261194614
Fold 4 IBS: 0.22387533549578809
Fold 5 IBS: 0.2083271491005745
[I 2024-04-13 17:20:13,120] Trial 60 finished with value: 0.22130694822708522 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 74, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.7017596527420207, 'min_weight_fraction_leaf': 0.09966382421080089}. Best is trial 46 with value: 0.21852635889761127.
Fold 1 IBS: 0.23644800149579903
Fold 2 IBS: 0.2195369287078321
Fold 3 IBS: 0.20653996690015303
Fold 4 IBS: 0.22236489358251474
Fold 5 IBS: 0.20806793260763023
[I 2024-04-13 17:20:20,677] Trial 61 finished with value: 0.21859154465878583 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 18, 'max_depth': 1, 'n_estimators': 357, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 

Fold 1 IBS: 0.23757083460058645
Fold 2 IBS: 0.21936835681341144
Fold 3 IBS: 0.2050586023900099
Fold 4 IBS: 0.2231460133461373
Fold 5 IBS: 0.20844072705559955
[I 2024-04-13 17:21:58,337] Trial 75 finished with value: 0.21871690684114892 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 20, 'min_samples_leaf': 19, 'max_depth': 2, 'n_estimators': 220, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.7803578114864529, 'min_weight_fraction_leaf': 0.017478919548903803}. Best is trial 73 with value: 0.21838002210423152.
Fold 1 IBS: 0.23640141704565878
Fold 2 IBS: 0.2197885256064894
Fold 3 IBS: 0.20318516882698215
Fold 4 IBS: 0.22544681939994865
Fold 5 IBS: 0.20897452087659646
[I 2024-04-13 17:22:03,878] Trial 76 finished with value: 0.21875929035113512 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 20, 'min_samples_leaf': 20, 'max_depth': 3, 'n_estimators': 293, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples'

Fold 1 IBS: 0.237217800059635
Fold 2 IBS: 0.219560418627486
Fold 3 IBS: 0.2055584995526966
Fold 4 IBS: 0.22351950054338868
Fold 5 IBS: 0.20798078328178596
[I 2024-04-13 17:23:15,144] Trial 90 finished with value: 0.21876740041299841 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 11, 'max_depth': 5, 'n_estimators': 306, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6578960771293777, 'min_weight_fraction_leaf': 0.16477439734294264}. Best is trial 73 with value: 0.21838002210423152.
Fold 1 IBS: 0.236951540585596
Fold 2 IBS: 0.21950546902732596
Fold 3 IBS: 0.20504430850227562
Fold 4 IBS: 0.22315671121676062
Fold 5 IBS: 0.20782359042139878
[I 2024-04-13 17:23:18,706] Trial 91 finished with value: 0.21849632395067142 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 8, 'max_depth': 6, 'n_estimators': 282, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.69

In [70]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [71]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.748
train_ibs:  0.218


#### Test

In [72]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [73]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=11, max_leaf_nodes=17,
                   max_samples=0.8662742964997368, min_samples_leaf=1,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.0017689713740798967,
                   n_estimators=54, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.585


ExtraSurvivalTrees(max_depth=2, max_features='auto', max_leaf_nodes=20,
                   max_samples=0.7842377375661996, min_samples_leaf=19,
                   min_samples_split=19,
                   min_weight_fraction_leaf=0.04843949872109438,
                   n_estimators=291, oob_score=True, random_state=123)

IBS: 0.213


In [74]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [75]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:23:54,731] A new study created in memory with name: no-name-56c55ad2-e968-4269-a63f-cbd3495e06a7


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:24:24,676] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:24:44,828] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:35:37,273] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.6288581469840132.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 17:37:29,853] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:00:54,229] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 12 with value: 0.6288581469840132.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:04:02,256] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'friedman_mse', 'ccp_alpha': 0.11820935501930148, 'min_weight_fraction

...

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:28:51,124] Trial 55 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.804828042321909, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.23926982708415356, 'n_estimators': 162, 'criterion': 'squared_error', 'ccp_alpha': 0.40074886285106287, 'min_weight_fraction_leaf': 0.2779066644542128, 'max_features': 'sqrt', 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 4, 'max_leaf_nodes': 16, 'min_samples_leaf': 19, 'max_depth': 4}. Best is trial 45 with value: 0.2332431097060344.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:29:21,922] Trial 56 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6106355356441384, 'learning_rate': 0.0175381503002972, 'dropout_rate': 0.184039934

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:32:01,699] Trial 66 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.9043457557527063, 'learning_rate': 0.08720647345857328, 'dropout_rate': 0.27193939046019544, 'n_estimators': 154, 'criterion': 'squared_error', 'ccp_alpha': 1.4111498627316026, 'min_weight_fraction_leaf': 0.23517339076530247, 'max_features': 'sqrt', 'min_impurity_decrease': 6.086951842225704e-07, 'validation_fraction': 0.9127218528145804, 'min_samples_split': 4, 'max_leaf_nodes': 15, 'min_samples_leaf': 20, 'max_depth': 2}. Best is trial 63 with value: 0.23223812135106253.
Fold 1 IBS: 0.24724710044658996
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:32:04,430] Trial 67 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.8250695857763196, 'learning_rate': 0.0911086342832341, 'dropout_rate': 0.3723000

Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:35:42,151] Trial 77 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7428517276581355, 'learning_rate': 0.07706989744762442, 'dropout_rate': 0.4476209058480567, 'n_estimators': 91, 'criterion': 'squared_error', 'ccp_alpha': 1.5888723787914292, 'min_weight_fraction_leaf': 0.16206442611838873, 'max_features': 0.1, 'min_impurity_decrease': 3.33671674086432e-07, 'validation_fraction': 0.9402673106586569, 'min_samples_split': 12, 'max_leaf_nodes': 13, 'min_samples_leaf': 17, 'max_depth': 9}. Best is trial 63 with value: 0.23223812135106253.
Fold 1 IBS: 0.24724710044658998
Fold 2 IBS: 0.23203988453792299
Fold 3 IBS: 0.22898186806977705
Fold 4 IBS: 0.24197477145927113
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:36:01,816] Trial 78 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.7715268987373948, 'learning_rate': 0.08985503754923349, 'dropout_rate': 0.51064243866

Fold 4 IBS: 0.24197477145927118
Fold 5 IBS: 0.2293955930480925
[I 2024-04-13 20:38:19,956] Trial 88 finished with value: 0.23592784351233073 and parameters: {'subsample': 0.6413430388178152, 'learning_rate': 0.0973528090376961, 'dropout_rate': 0.26912242502602596, 'n_estimators': 174, 'criterion': 'squared_error', 'ccp_alpha': 0.5807061908097002, 'min_weight_fraction_leaf': 0.16723537370276506, 'max_features': 0.1, 'min_impurity_decrease': 1.190634063232502e-06, 'validation_fraction': 0.8005899065401637, 'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 17, 'max_depth': 4}. Best is trial 82 with value: 0.23205660370863482.
Fold 1 IBS: 0.24220714862378384
Fold 2 IBS: 0.22705284477038132
Fold 3 IBS: 0.22374120154251026
Fold 4 IBS: 0.23761138561741843
Fold 5 IBS: 0.22312495335199894
[I 2024-04-13 20:38:43,138] Trial 89 finished with value: 0.23074750678121858 and parameters: {'subsample': 0.38730242337646104, 'learning_rate': 0.08109800836387941, 'dropout_rate': 0.18122999

Fold 4 IBS: 0.23852342496708018
Fold 5 IBS: 0.22486084489133845
[I 2024-04-13 20:42:22,257] Trial 99 finished with value: 0.23142156664291713 and parameters: {'subsample': 0.40791967441342253, 'learning_rate': 0.07712333235501795, 'dropout_rate': 0.19859974199600133, 'n_estimators': 143, 'criterion': 'squared_error', 'ccp_alpha': 0.004426170739568851, 'min_weight_fraction_leaf': 0.16314282359709542, 'max_features': 'log2', 'min_impurity_decrease': 5.353566353972388e-07, 'validation_fraction': 0.1604355124738367, 'min_samples_split': 11, 'max_leaf_nodes': 15, 'min_samples_leaf': 14, 'max_depth': 4}. Best is trial 91 with value: 0.22930168836973525.


* Best trial for IBS: 
 FrozenTrial(number=91, state=TrialState.COMPLETE, values=[0.22930168836973525], datetime_start=datetime.datetime(2024, 4, 13, 20, 39, 24, 831303), datetime_complete=datetime.datetime(2024, 4, 13, 20, 39, 49, 113073), params={'subsample': 0.3646963250854607, 'learning_rate': 0.09255241886126767, 'dropout_rate': 0.1236

In [76]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [77]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.648
train_ibs:  0.229


#### Test

In [78]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [79]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.02647951541473475,
                                 criterion='squared_error',
                                 dropout_rate=0.14303111133639826,
                                 learning_rate=0.05317196534413593,
                                 max_depth=19, max_features=0.1,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=7.356647091617949e-06,
                                 min_samples_leaf=17, min_samples_split=19,
                                 min_weight_fraction_leaf=0.35066087197674667,
                                 n_estimators=78, random_state=123,
                                 subsample=0.9736544653699906,
                                 validation_fraction=0.6117331205297228)

C-index score: 0.639


GradientBoostingSurvivalAnalysis(ccp_alpha=0.03832008885345805,
                                 criterion='squared_error',
                                 dropout_rate=0.12363276406693596,
                                 learning_rate=0.09255241886126767, max_depth=4,
                                 max_leaf_nodes=15,
                                 min_impurity_decrease=6.12703415921239e-07,
                                 min_samples_leaf=16, min_samples_split=13,
                                 min_weight_fraction_leaf=0.14198218649980035,
                                 n_estimators=143, random_state=123,
                                 subsample=0.3646963250854607,
                                 validation_fraction=0.965806971606102)

IBS: 0.221


In [80]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [81]:
# Setting the y format 
y = clinical_train[['DFS', 'event_DFS']]

In [82]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_DFS'], y_train_df['DFS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_DFS'], y_test_df['DFS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 20:42:27,329] A new study created in memory with name: no-name-7d3b3870-f3c9-4a92-a4f8-712a3e3eb8f7


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.501937984496124
Fold 3 C-index: 0.5553191489361702
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.6201716738197425
[I 2024-04-13 20:42:29,632] Trial 0 finished with value: 0.5612882548986675 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.5612882548986675.
Fold 1 C-index: 0.5358565737051793
Fold 2 C-index: 0.49806201550387597
Fold 3 C-index: 0.5531914893617021
Fold 4 C-index: 0.5931558935361216
Fold 5 C-index: 0.6244635193133047
[I 2024-04-13 20:42:48,458] Trial 1 finished with value: 0.5609458982840367 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.5612882548986675.
Fold 1 C-index: 0.599601593625498
Fold 2 C-index: 0.501937984496124
Fold 3 C-index: 0.5595744680851064
Fold 4 

Fold 1 C-index: 0.6175298804780877
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5723404255319149
Fold 4 C-index: 0.5855513307984791
Fold 5 C-index: 0.6394849785407726
[I 2024-04-13 20:45:05,049] Trial 19 finished with value: 0.5849193075659749 and parameters: {'subsample': 0.3739805428810522, 'dropout_rate': 0.6293352695898036, 'n_estimators': 295, 'learning_rate': 0.09107738339755381}. Best is trial 15 with value: 0.6193869827160372.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5271317829457365
Fold 3 C-index: 0.6021276595744681
Fold 4 C-index: 0.6539923954372624
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 20:45:18,039] Trial 20 finished with value: 0.6089732648436864 and parameters: {'subsample': 0.179916923106687, 'dropout_rate': 0.5777838682216713, 'n_estimators': 416, 'learning_rate': 0.06551866052750378}. Best is trial 15 with value: 0.6193869827160372.
Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.5310077519379846
Fold 3 C-index: 0.6319148936170212
Fol

Fold 1 C-index: 0.6055776892430279
Fold 2 C-index: 0.5096899224806202
Fold 3 C-index: 0.5765957446808511
Fold 4 C-index: 0.5779467680608364
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 20:47:10,073] Trial 38 finished with value: 0.5835757587986465 and parameters: {'subsample': 0.32929695725910635, 'dropout_rate': 0.3720991365773877, 'n_estimators': 188, 'learning_rate': 0.09321185753440044}. Best is trial 33 with value: 0.6209005899946404.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5193798449612403
Fold 3 C-index: 0.5851063829787234
Fold 4 C-index: 0.6311787072243346
Fold 5 C-index: 0.648068669527897
[I 2024-04-13 20:47:19,136] Trial 39 finished with value: 0.5994558842850526 and parameters: {'subsample': 0.2296067526961567, 'dropout_rate': 0.8600360752966183, 'n_estimators': 367, 'learning_rate': 0.0777390157253494}. Best is trial 33 with value: 0.6209005899946404.
Fold 1 C-index: 0.6135458167330677
Fold 2 C-index: 0.5271317829457365
Fold 3 C-index: 0.6319148936170212
Fol

...

Fold 3 IBS: 0.22471703387329683
Fold 4 IBS: 0.2514223976299201
Fold 5 IBS: 0.209166894601268
[I 2024-04-13 20:54:55,410] Trial 19 finished with value: 0.23609624158230963 and parameters: {'subsample': 0.3892284838807411, 'dropout_rate': 0.5354432466556798, 'n_estimators': 131, 'learning_rate': 0.023294697221931306}. Best is trial 7 with value: 0.2317185873300352.
Fold 1 IBS: 0.2446002185537661
Fold 2 IBS: 0.23463757481110972
Fold 3 IBS: 0.2201655501901051
Fold 4 IBS: 0.23919942537593147
Fold 5 IBS: 0.22296335334244682
[I 2024-04-13 20:54:56,699] Trial 20 finished with value: 0.23231322445467187 and parameters: {'subsample': 0.6210873354088753, 'dropout_rate': 0.7933084651006226, 'n_estimators': 66, 'learning_rate': 0.013007963411749002}. Best is trial 7 with value: 0.2317185873300352.
Fold 1 IBS: 0.24513654608372853
Fold 2 IBS: 0.23383949946417815
Fold 3 IBS: 0.22211599951175628
Fold 4 IBS: 0.24019573213746584
Fold 5 IBS: 0.22475950483168106
[I 2024-04-13 20:54:57,964] Trial 21 finishe

Fold 4 IBS: 0.23734359543462524
Fold 5 IBS: 0.2194438033958121
[I 2024-04-13 20:55:49,250] Trial 38 finished with value: 0.23170597177038704 and parameters: {'subsample': 0.6892895892485054, 'dropout_rate': 0.3454528205709918, 'n_estimators': 302, 'learning_rate': 0.00460239710283736}. Best is trial 38 with value: 0.23170597177038704.
Fold 1 IBS: 0.31972383764115336
Fold 2 IBS: 0.3786035551717951
Fold 3 IBS: 0.31938841247246336
Fold 4 IBS: 0.3506943227141584
Fold 5 IBS: 0.337145270635712
[I 2024-04-13 20:55:57,744] Trial 39 finished with value: 0.3411110797270564 and parameters: {'subsample': 0.5208265537491429, 'dropout_rate': 0.3158258870928942, 'n_estimators': 294, 'learning_rate': 0.07379222544900951}. Best is trial 38 with value: 0.23170597177038704.
Fold 1 IBS: 0.24434143323606985
Fold 2 IBS: 0.2415994203280167
Fold 3 IBS: 0.21847616164629763
Fold 4 IBS: 0.23711394107358877
Fold 5 IBS: 0.21776977306596226
[I 2024-04-13 20:56:09,928] Trial 40 finished with value: 0.231860145869987

Fold 4 IBS: 0.3243986654312856
Fold 5 IBS: 0.26528172110872605
[I 2024-04-13 20:59:39,003] Trial 57 finished with value: 0.30401873031615184 and parameters: {'subsample': 0.2888110376335854, 'dropout_rate': 0.2718615358996285, 'n_estimators': 342, 'learning_rate': 0.03512375325073246}. Best is trial 44 with value: 0.23142014649076392.
Fold 1 IBS: 0.25755648343454
Fold 2 IBS: 0.2764498175883411
Fold 3 IBS: 0.24200617130873298
Fold 4 IBS: 0.2432893725992331
Fold 5 IBS: 0.21437684756519773
[I 2024-04-13 20:59:45,703] Trial 58 finished with value: 0.24673573849920896 and parameters: {'subsample': 0.6878065235587841, 'dropout_rate': 0.5792181458125746, 'n_estimators': 278, 'learning_rate': 0.01539553219464254}. Best is trial 44 with value: 0.23142014649076392.
Fold 1 IBS: 0.2450244777809273
Fold 2 IBS: 0.2459114617756686
Fold 3 IBS: 0.2196599956493579
Fold 4 IBS: 0.23673683057657782
Fold 5 IBS: 0.21651218141381678
[I 2024-04-13 20:59:51,199] Trial 59 finished with value: 0.2327689894392697 

Fold 4 IBS: 0.24101925396002535
Fold 5 IBS: 0.2179470573597778
[I 2024-04-13 21:02:01,253] Trial 76 finished with value: 0.23125325458142493 and parameters: {'subsample': 0.4955197477745412, 'dropout_rate': 0.16046855961245599, 'n_estimators': 195, 'learning_rate': 0.007706860559248215}. Best is trial 64 with value: 0.23119183671299623.
Fold 1 IBS: 0.24421634647346885
Fold 2 IBS: 0.23548728538557162
Fold 3 IBS: 0.21781247557561054
Fold 4 IBS: 0.2416619653489052
Fold 5 IBS: 0.21716988899584236
[I 2024-04-13 21:02:05,374] Trial 77 finished with value: 0.2312695923558797 and parameters: {'subsample': 0.4935643600806539, 'dropout_rate': 0.15685680514657221, 'n_estimators': 189, 'learning_rate': 0.008564077446745044}. Best is trial 64 with value: 0.23119183671299623.
Fold 1 IBS: 0.25044273628462804
Fold 2 IBS: 0.24911955068637429
Fold 3 IBS: 0.23186848614200992
Fold 4 IBS: 0.2564736956596988
Fold 5 IBS: 0.2063404965895953
[I 2024-04-13 21:02:09,557] Trial 78 finished with value: 0.238848993

Fold 3 IBS: 0.21965424337510803
Fold 4 IBS: 0.238100479483417
Fold 5 IBS: 0.21419247900632338
[I 2024-04-13 21:03:15,161] Trial 95 finished with value: 0.23131233350275093 and parameters: {'subsample': 0.5287900994881528, 'dropout_rate': 0.24362538801389683, 'n_estimators': 114, 'learning_rate': 0.01918720977286184}. Best is trial 91 with value: 0.23094053868721942.
Fold 1 IBS: 0.24450793825144282
Fold 2 IBS: 0.23721942064100832
Fold 3 IBS: 0.2191361464016592
Fold 4 IBS: 0.23967108610942914
Fold 5 IBS: 0.21529353238617982
[I 2024-04-13 21:03:18,834] Trial 96 finished with value: 0.23116562475794383 and parameters: {'subsample': 0.5479763862885562, 'dropout_rate': 0.18732015009820935, 'n_estimators': 159, 'learning_rate': 0.012590794461904201}. Best is trial 91 with value: 0.23094053868721942.
Fold 1 IBS: 0.24460738141356492
Fold 2 IBS: 0.24083856752760174
Fold 3 IBS: 0.21870952652926662
Fold 4 IBS: 0.23884101095979912
Fold 5 IBS: 0.21590250993624882
[I 2024-04-13 21:03:22,805] Trial 97

In [83]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [84]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.627
train_ibs:  0.231


#### Test

In [85]:
# y into array 
lists = [] 
for i, j in zip(y['event_DFS'], y['DFS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [86]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10355008884313799,
                                              learning_rate=0.09604343153434695,
                                              n_estimators=308,
                                              random_state=123,
                                              subsample=0.10803471257567032)

C-index score: 0.567


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.2314439741310604,
                                              learning_rate=0.011421014743335248,
                                              n_estimators=139,
                                              random_state=123,
                                              subsample=0.5279987131107713)

IBS: 0.243


In [87]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [88]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.790,1.0
ExtraSurvivalTrees,0.748,2.0
GradientBoosting,0.648,3.0
CoxPH,0.638,4.5
CoxElastic,0.638,4.5
CoxLasso,0.637,6.0
ComponentwiseGradientBoosting,0.627,7.0
CoxRidge,0.583,8.0


In [89]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.218,1.5
ExtraSurvivalTrees,0.218,1.5
CoxElastic,0.227,3.0
GradientBoosting,0.229,4.0
CoxPH,0.231,6.0
CoxLasso,0.231,6.0
ComponentwiseGradientBoosting,0.231,6.0
CoxRidge,0.236,8.0


In [90]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.639,1.0
Randomsurvivalforest,0.603,2.0
ExtraSurvivalTrees,0.585,3.0
CoxRidge,0.571,4.0
ComponentwiseGradientBoosting,0.567,5.0
CoxPH,0.532,7.0
CoxLasso,0.532,7.0
CoxElastic,0.532,7.0


In [91]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
ExtraSurvivalTrees,0.213,1.0
Randomsurvivalforest,0.215,2.0
GradientBoosting,0.221,3.0
CoxRidge,0.229,4.0
ComponentwiseGradientBoosting,0.243,5.0
CoxLasso,0.273,6.5
CoxElastic,0.273,6.5
CoxPH,0.276,8.0


In [92]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/dfs/robust/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_dfs_robust_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [93]:
from datetime import date
today = date.today()
print("Date: ", today)


Date:  2024-04-13
